In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

from face_utils import compute_pca, compute_mda, separate_train_test_manual, plot_classification_results

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
data["face"].shape

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
len(data_classified[1])

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


## bayes classifier

In [ ]:
import numpy as np

def bayes_classifier(train_set, train_labels, test_set, test_labels, num_classes):
    """
    Implements Bayes classifier with Gaussian assumption (QDA-style).
    Parameters:
        train_set: np.ndarray, shape (n_train_samples, n_features)
        train_labels: np.ndarray, shape (n_train_samples,)
        test_set: np.ndarray, shape (n_test_samples, n_features)
        test_labels: np.ndarray, shape (n_test_samples,)
        num_classes: int — number of classes (2 for expression task)
    Returns:
        y_pred: predicted class labels
        accuracy: classification accuracy in percentage
    """
    n_features = train_set.shape[1]
    class_means = {}  # Use dictionary instead of array
    class_covs = {}   # Use dictionary instead of array
    
    # Step 1: Compute MLE of mean and covariance for each class
    for c in range(num_classes):
        class_data = train_set[train_labels == c]
        
        # Check if there are any samples for this class
        if len(class_data) > 0:
            class_means[c] = np.mean(class_data, axis=0)
            
            sigma = np.zeros((n_features, n_features))
            for sample in class_data:
                diff = sample - class_means[c]
                sigma += np.outer(diff, diff)
            class_covs[c] = (sigma / len(class_data)) + np.eye(n_features) * 0.92  # Regularization
        else:
            print(f"Warning: Class {c} not found in training data")
    
    # Step 2: Evaluate discriminant function for each test point
    valid_classes = list(class_means.keys())
    y_pred = np.zeros(len(test_set), dtype=int)

    # Invert each class covariance once; it does not depend on the test point
    cov_invs = {c: np.linalg.inv(class_covs[c]) for c in valid_classes}
    cov_dets = {c: np.linalg.det(class_covs[c]) for c in valid_classes}
    
    for i, x in enumerate(test_set):
        g = {}  # Use dictionary to store discriminant values
        
        for c in valid_classes:
            cov_inv = cov_invs[c]
            cov_det = cov_dets[c]
            mu = class_means[c]
            
            quad_term = -0.5 * np.dot(np.dot(x, cov_inv), x)
            linear_term = np.dot(mu, np.dot(cov_inv, x))
            const_term = -0.5 * np.dot(np.dot(mu, cov_inv), mu) - 0.5 * np.log(cov_det)
            
            g[c] = quad_term + linear_term + const_term
        
        # Handle the case where no valid classes exist
        if g:
            y_pred[i] = max(g, key=g.get)
        else:
            y_pred[i] = 0  # Default prediction if no valid classes
    
    accuracy = np.mean(y_pred == test_labels) * 100
    accuracy2 = (sum(y_pred == test_labels)/len(test_labels))*100
    accuracy3 = (len(y_pred == test_labels)/len(test_labels))*100
    return y_pred, accuracy, accuracy2, accuracy3

In [ ]:
import matplotlib.pyplot as plt

def plot_bayes_output_vs_labels(y_pred, y_true, task_num=1):
    plt.figure(figsize=(12, 4))

    plt.plot(y_pred, 'o', label="Bayes output", linewidth=2.8, color='blue')
    plt.plot(y_true, '*', label="Labels", linewidth=1.0, color='red')

    plt.title("Classifying Neutral and Expression using Bayes classifier on dataset DATA")
    plt.xlabel("Test sample index")
    plt.ylabel("Class label")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# import matplotlib.pyplot as plt
import numpy as np

# Task 1

In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,
    task_num=1
)

In [ ]:
pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=139)

# 4. Project test set using the same PCA basis
centered_test = test_set - mean_vec
pca_result_test = np.dot(centered_test, components)

In [ ]:
mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=199)

# 4. Project test set using the same MDA basis
centered_test = test_set - mean_vec
mda_result_test = np.dot(centered_test, components)

In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(train_set, train_labels, test_set, test_labels, num_classes=200)
print(f"✅ Person Identification Accuracy (Bayes): {accuracy:.2f}%")


In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=200)
print(f"✅ Person Identification Accuracy (Bayes, PCA): {accuracy:.2f}%")


In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(mda_result_train, train_labels, mda_result_test, test_labels, num_classes=200)
print(f"✅ Person Identification Accuracy (Bayes, MDA): {accuracy:.2f}%")


In [ ]:
plot_bayes_output_vs_labels(y_pred, test_labels, task_num=1)
plot_classification_results(y_pred, test_labels, task_num=1)

In [ ]:
import matplotlib.pyplot as plt

train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,
    task_num=1
)

# Define component ranges
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
mda_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139, 150, 170, 199]

# Store results
pca_accuracies = {}
mda_accuracies = {}

# Run PCA experiments
for num_components in pca_components_list:
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=num_components)
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)
    
    _, accuracy, _, _ = bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=200)
    pca_accuracies[num_components] = accuracy
    print(f"PCA {num_components} components → Accuracy: {accuracy:.2f}%")

# Run MDA experiments
for num_components in mda_components_list:
    mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=num_components)
    centered_test = test_set - mean_vec
    mda_result_test = np.dot(centered_test, components)
    
    _, accuracy, _, _ = bayes_classifier(mda_result_train, train_labels, mda_result_test, test_labels, num_classes=200)
    mda_accuracies[num_components] = accuracy
    print(f"MDA {num_components} components → Accuracy: {accuracy:.2f}%")

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(list(pca_accuracies.keys()), list(pca_accuracies.values()), marker='o', label='PCA + Bayes')
plt.plot(list(mda_accuracies.keys()), list(mda_accuracies.values()), marker='s', label='MDA + Bayes')

plt.title('Classification Accuracy vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
plt.plot(list(pca_accuracies.keys()), list(pca_accuracies.values()), marker='o', label='PCA + Bayes')
plt.plot(list(mda_accuracies.keys()), list(mda_accuracies.values()), marker='s', label='MDA + Bayes')
plt.axhline(64.00, color ='red',linestyle='--', label=f'normal bayes: 64.00%' )

plt.title('Classification Accuracy task 1 vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Task 2

In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,  # Use all subjects
    task_num=2
)

len(test_set), len(train_set)

In [ ]:
pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=139)

# 4. Project test set using the same PCA basis
centered_test = test_set - mean_vec
pca_result_test = np.dot(centered_test, components)

In [ ]:
mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=1)

# 4. Project test set using the same MDA basis
centered_test = test_set - mean_vec
mda_result_test = np.dot(centered_test, components)

In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(train_set, train_labels, test_set, test_labels, num_classes=2)
print(f"✅ Expression Classification Accuracy (Bayes): {accuracy:.2f}%")


In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=2)
print(f"✅ Expression Classification Accuracy (Bayes, PCA): {accuracy:.2f}%")


In [ ]:
plot_bayes_output_vs_labels(y_pred, test_labels, task_num=1)
plot_classification_results(y_pred, test_labels, task_num=1)

In [ ]:
components_number=[1,2,3,10,50,80,100,120,139]
pca_accuracies={}

for i in components_number: 
    
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=i)

    # 4. Project test set using the same PCA basis
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)
    
    # Step 4: Run Bayes classifier
    y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(pca_result_train, train_labels, pca_result_test, test_labels, num_classes=2)
    print(f"✅ Expression Classification Accuracy (Bayes, PCA): {accuracy:.2f}%")
    pca_accuracies[i]=accuracy

In [ ]:
# Step 4: Run Bayes classifier
y_pred, accuracy ,accuracy2, accuracy3= bayes_classifier(mda_result_train, train_labels, mda_result_test, test_labels, num_classes=2)
print(f"✅ Expression Classification Accuracy (Bayes, MDA): {accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Given PCA accuracies from your printouts
components_number = [1, 2, 3, 10, 50, 80, 100, 120, 139]
pca_accuracies = {
    1: 62.5,
    2: 58.75,
    3: 75.0,
    10: 80.0,
    50: 78.75,
    80: 81.25,
    100: 81.25,
    120 :80.00, 
    139: 78.75
}

# MDA accuracy
mda_accuracy = 92.5

# regular_bayes
normal_bayes = 78.75

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(components_number, [pca_accuracies[k] for k in components_number], marker='o', label='Bayes (PCA)')
plt.axhline(mda_accuracy, color='red', linestyle='--', label=f'Bayes (MDA): {mda_accuracy:.2f}%')
plt.axhline(normal_bayes, color ='orange',linestyle='--', label=f'normal bayes: {normal_bayes:.2f}%' )

plt.title('Expression Classification Accuracy task 2')
plt.xlabel('Number of PCA Components')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
